# 🧹 04 - Lakehouse Table Maintenance (Compaction & Expiry)

### 📌 Objective
In this notebook, we run production table maintenance procedures in Apache Iceberg using Spark SQL system procedures:
1. **Compaction (`rewrite_data_files`):** Solves the "small files problem" by combining tiny Parquet files into optimal sizes.
2. **Snapshot Expiration (`expire_snapshots`):** Purges unreferenced historical files to keep metadata lean and lower S3 storage costs.

### 💡 Why Table Maintenance is Mandatory in Production:
- Streaming or frequent batch appends generate thousands of small files.
- Query engines spend more time opening S3 connections for tiny files than reading actual data.
- Compaction rewrites small files in place using atomic replacement snapshots.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Step A: View snapshots BEFORE maintenance
print("=== SNAPSHOTS BEFORE MAINTENANCE ===")
spark.sql("""
    SELECT committed_at, snapshot_id, operation, summary['total-records'] as total_records, summary['total-data-files'] as total_files
    FROM icelake.wikipedia.articles.snapshots
    ORDER BY committed_at ASC
""").show(truncate=False)

### Step 1: Run Data File Compaction (`rewrite_data_files`)
Iceberg's `rewrite_data_files` uses a **binpack** or **sort** strategy to combine small files without blocking concurrent reads.

In [ ]:
print("🧹 Running Compaction (rewrite_data_files)...")
result = spark.sql("""
    CALL icelake.system.rewrite_data_files(
        table => 'wikipedia.articles',
        strategy => 'binpack',
        options => map('min-input-files', '1')
    )
""")
result.show(truncate=False)

### Step 2: Expire Old Snapshots (`expire_snapshots`)
We retain only the latest active snapshot, pruning metadata references.

In [ ]:
print("⏳ Expiring Old Snapshots...")
result = spark.sql("""
    CALL icelake.system.expire_snapshots(
        table => 'wikipedia.articles',
        retain_last => 2
    )
""")
result.show(truncate=False)

### Step 3: View Snapshots After Maintenance

In [ ]:
print("=== SNAPSHOTS AFTER MAINTENANCE ===")
spark.sql("""
    SELECT committed_at, snapshot_id, operation, summary['total-records'] as total_records
    FROM icelake.wikipedia.articles.snapshots
    ORDER BY committed_at ASC
""").show(truncate=False)